# Fine-tune SPECTER2 on DePaul Faculty Data

**Before running anything:** `Runtime → Change runtime type → T4 GPU`

Then run each cell top to bottom. When Cell 3 runs, it will ask you to upload a file — upload `data/training_pairs.json` from your local project.

Total time: ~20-30 minutes on T4 GPU.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q sentence-transformers datasets adapters

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Cell 2 — Upload training data
# Upload your local file:  data/training_pairs.json
from google.colab import files
import json

print('Upload your data/training_pairs.json file when the dialog appears...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
train_pairs = json.loads(uploaded[filename])
print(f'Loaded {len(train_pairs)} training pairs from {len({p["faculty_id"] for p in train_pairs})} faculty')

In [ ]:
# Cell 3 — Extract unique faculty bios for evaluation
seen_ids = set()
faculty_bios = []
for p in train_pairs:
    if p['faculty_id'] not in seen_ids:
        seen_ids.add(p['faculty_id'])
        faculty_bios.append((p['faculty_name'], p['bio']))

print(f'Evaluation corpus: {len(faculty_bios)} unique faculty bios')
print(f'Sample: {faculty_bios[0][0]} — {faculty_bios[0][1][:80]}...')

In [ ]:
# Cell 4 — Define test queries (mix of technical + lay language)
TEST_QUERIES = [
    {"query": "Alzheimer disease neurodegeneration amyloid protein",
     "expected": ["Eric Norstrom", "Eiron Cudaback"]},
    {"query": "machine learning clinical decision support healthcare",
     "expected": ["Casey Bennett"]},
    {"query": "natural language processing text classification sentiment",
     "expected": ["Noriko Tomuro"]},
    {"query": "network security intrusion detection anomaly",
     "expected": ["Jean-Philippe Labruyere"]},
    {"query": "deep learning computer vision image segmentation",
     "expected": ["Daniela Stan Raicu"]},
    # Lay-language queries — these test real-world user behaviour
    {"query": "memory loss in elderly patients",
     "expected": ["Eric Norstrom", "Eiron Cudaback"]},
    {"query": "finding early signs of Alzheimer in old people",
     "expected": ["Eric Norstrom", "Eiron Cudaback"]},
    {"query": "AI that explains its own decisions",
     "expected": []},
    {"query": "law process in criminal charges",
     "expected": []},  # should NOT return CS faculty
]

print(f'{len(TEST_QUERIES)} test queries ({len([q for q in TEST_QUERIES if q["expected"]])} with expected results)')

In [ ]:
# Cell 5 — Evaluation function
import numpy as np

def evaluate(model, faculty_bios, test_queries, label=''):
    names    = [name for name, _ in faculty_bios]
    bios     = [bio  for _, bio  in faculty_bios]
    bio_embs = model.encode(bios, normalize_embeddings=True,
                             show_progress_bar=False, batch_size=64)
    scores_list = []
    for item in test_queries:
        q        = item['query']
        expected = item.get('expected', [])
        if not expected:
            # Still show top3 so we can spot false positives
            qv   = model.encode([q], normalize_embeddings=True)[0]
            sims = bio_embs @ qv
            top3 = [names[i] for i in np.argsort(sims)[::-1][:3]]
            print(f'  -  [{q[:50]}]  top3={top3}')
            continue
        qv   = model.encode([q], normalize_embeddings=True)[0]
        sims = bio_embs @ qv
        top5 = np.argsort(sims)[::-1][:5]
        found = [names[i] for i in top5]
        hits  = sum(1 for name in expected if name in found)
        p5    = hits / max(len(expected), 1)
        scores_list.append(p5)
        status = 'v' if hits > 0 else 'x'
        print(f'  {status}  [{q[:50]}]  top3={found[:3]}')
    mean_p5 = sum(scores_list) / len(scores_list) if scores_list else 0.0
    print(f'  --> {label}: precision@5 = {mean_p5:.3f}  ({len(scores_list)} queries with expected results)')
    return mean_p5

print('Evaluation function ready.')

In [ ]:
# Cell 6 — Evaluate baseline (no fine-tuning)
from sentence_transformers import SentenceTransformer

print('=' * 60)
print('Baseline: allenai/specter2_base')
print('=' * 60)
baseline_model = SentenceTransformer('allenai/specter2_base', device='cuda')
baseline_p5    = evaluate(baseline_model, faculty_bios, TEST_QUERIES, label='baseline')

del baseline_model
torch.cuda.empty_cache()

In [ ]:
# Cell 7 — Fine-tune 3 configs and compare
import os, time
from sentence_transformers import (
    SentenceTransformer, SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from datasets import Dataset

CONFIGS = [
    {'name': 'A_fast',         'epochs': 1, 'lr': 2e-5, 'batch': 32},
    {'name': 'B_standard',     'epochs': 3, 'lr': 2e-5, 'batch': 32},
    {'name': 'C_conservative', 'epochs': 3, 'lr': 5e-6, 'batch': 32},
]

results      = {'baseline': baseline_p5}
saved_models = {}

for cfg in CONFIGS:
    name, epochs, lr, batch = cfg['name'], cfg['epochs'], cfg['lr'], cfg['batch']
    out = f'/content/models/specter2_depaul_{name}'
    os.makedirs('/content/models', exist_ok=True)

    print(f'\n{"=" * 60}')
    print(f'Config {name}: {epochs} epoch(s), lr={lr}, batch={batch}')
    print(f'{"=" * 60}')

    model = SentenceTransformer('allenai/specter2_base', device='cuda')
    loss  = MultipleNegativesRankingLoss(model)

    train_dataset = Dataset.from_dict({
        'anchor':   [p['query'] for p in train_pairs],
        'positive': [p['bio']   for p in train_pairs],
    })

    train_args = SentenceTransformerTrainingArguments(
        output_dir=out,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        learning_rate=lr,
        save_strategy='no',
        fp16=True,
    )

    start   = time.time()
    trainer = SentenceTransformerTrainer(
        model=model, args=train_args,
        train_dataset=train_dataset, loss=loss
    )
    trainer.train()
    model.save(out)
    elapsed = time.time() - start
    print(f'  Training done in {elapsed/60:.1f} min')

    p5 = evaluate(model, faculty_bios, TEST_QUERIES, label=name)
    results[name]      = p5
    saved_models[name] = out

    del model
    torch.cuda.empty_cache()

# Summary
print(f'\n{"=" * 60}')
print('FINAL COMPARISON  (precision@5)')
print(f'{"=" * 60}')
best_name = max(results, key=results.get)
for name, p5 in results.items():
    marker = '  <-- BEST' if name == best_name else ''
    print(f'  {name:25s}  {p5:.3f}{marker}')

In [ ]:
# Cell 8 — Download the best fine-tuned model
import shutil
from google.colab import files

# Pick the best non-baseline config
best_cfg  = max((n for n in results if n != 'baseline'), key=lambda n: results[n])
best_path = saved_models[best_cfg]
zip_path  = f'/content/specter2_depaul_{best_cfg}'

print(f'Best config: {best_cfg}  (precision@5 = {results[best_cfg]:.3f})')

if results[best_cfg] > results['baseline']:
    print(f'Improvement over baseline: +{results[best_cfg] - results["baseline"]:.3f}')
else:
    print('Note: fine-tuned model did not beat baseline on these queries.')
    print('Still downloading — it may perform better on lay-language queries.')

print(f'Zipping {best_path} ...')
shutil.make_archive(zip_path, 'zip', best_path)
print('Downloading ...')
files.download(f'{zip_path}.zip')

print(f'''
After download, on your local machine:
  mkdir -p models/specter2_depaul_{best_cfg}
  unzip specter2_depaul_{best_cfg}.zip -d models/specter2_depaul_{best_cfg}/
  export FINETUNED_MODEL=models/specter2_depaul_{best_cfg}
  rm -f faculty_index.pkl && python3 search.py
''')